In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 相对 notebook 位置定位 CSV，无论从哪里启动都能找到
LOG_PATH = os.path.join(os.path.dirname(os.path.abspath("log_watcher.ipynb")),
                        "logs", "train_log.csv")
print("读取：", LOG_PATH)

In [ ]:
df = pd.read_csv(LOG_PATH)
print(f"已训练迭代：{len(df)}，最新 iter={df['iteration'].iloc[-1]}")

# Loss 曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[["value_loss", "policy_loss"]].plot(ax=axes[0], title="Loss 曲线")
axes[0].set_xlabel("迭代")
axes[0].grid(True)

# 胜率曲线（只取有评估数据的行）
win_cols = [c for c in df.columns if c.startswith("wins_")]
eval_df = df.dropna(subset=win_cols[:1]).copy()
if not eval_df.empty:
    eval_df[win_cols].plot(ax=axes[1], marker="o", title="胜率（每项满分 8 或 4）")
    axes[1].set_xlabel("index（eval 行）")
    axes[1].grid(True)
else:
    axes[1].set_title("暂无评估数据（等到 iter 50）")

plt.tight_layout()
plt.show()

In [ ]:
# 最近 5 条数字摘要
print(f"最近 value_loss ：{df['value_loss'].dropna().tail(5).values}")
print(f"最近 policy_loss：{df['policy_loss'].dropna().tail(5).values}")

win_cols = [c for c in df.columns if c.startswith("wins_")]
eval_df = df[df[win_cols[0]].notna()][["iteration"] + win_cols].tail(5) if win_cols else None
if eval_df is not None and not eval_df.empty:
    print("\n最近评估结果：")
    print(eval_df.to_string(index=False))